# LLM Impersonation – Interactive Exploration

This notebook allows you to:
1. Inspect Ayush's dataset (all 40 Q&A pairs)
2. Try each impersonation method interactively on a question of your choice
3. Compute and compare evaluation metrics on individual responses
4. Visualise results from `results/`

**Prerequisites:** Install dependencies first.
```bash
pip install -r requirements.txt
export OPENAI_API_KEY="your-key-here"
```

In [ ]:
import sys, pathlib
# Make sure the repo root is on the path
repo_root = pathlib.Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data_loader import load_train, load_val, load_test, load_full
import pandas as pd

full = load_full()
df = pd.DataFrame(full)
df

## 1. Dataset Overview

In [ ]:
print(f"Total pairs : {len(full)}")
print(f"Train pairs : {len(load_train())}")
print(f"Val pairs   : {len(load_val())}")
print(f"Test pairs  : {len(load_test())}")

# Category distribution
df['category'].value_counts()

## 2. Try the Methods Interactively

Set your OpenAI API key in the environment before running these cells.

In [ ]:
from src.utils import build_openai_client

client = build_openai_client()   # reads OPENAI_API_KEY from env
train_data = load_train()

In [ ]:
# ── Zero-shot ──────────────────────────────────────────────
from src.methods import zero_shot

question = "What's your opinion on climate change?"
response = zero_shot.generate(client, question)
print(f"Q: {question}")
print(f"A (zero-shot): {response}")

In [ ]:
# ── Few-shot ───────────────────────────────────────────────
from src.methods import few_shot

response = few_shot.generate(client, question, train_data=train_data)
print(f"Q: {question}")
print(f"A (few-shot): {response}")

In [ ]:
# ── RAG ────────────────────────────────────────────────────
from src.methods.rag import RAGIndex, generate as rag_generate

rag_index = RAGIndex(train_data)
response, retrieved = rag_generate(client, question, rag_index)
print(f"Q: {question}")
print(f"A (RAG): {response}")
print(f"\nRetrieved context:")
for r in retrieved:
    print(f"  Q{r['id']}: {r['question']} -> {r['answer']}")

## 3. Evaluate a Single Response

In [ ]:
from src.evaluation import rouge_eval, bert_score_eval, cosine_similarity_eval

# Example using ground-truth answer for Q31
test_item = {
    "id": 31,
    "question": question,
    "ground_truth": "Can be reversed if we reduce the carbon emissions",
    "generated": response,   # replace with any generated response
}

results = [test_item]
results = rouge_eval.evaluate(results)
results = bert_score_eval.evaluate(results)
results = cosine_similarity_eval.evaluate(results)

r = results[0]
print(f"ROUGE-L       : {r['rouge_l']}")
print(f"BERTScore F1  : {r['bertscore_f1']}")
print(f"Cosine Sim    : {r['cosine_similarity']}")

## 4. Visualise Experiment Results

Run `python experiments/run_all_experiments.py` first to generate `results/summary.json`.

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
import numpy as np

summary_path = repo_root / "results" / "summary.json"
if not summary_path.exists():
    print("Run run_all_experiments.py first.")
else:
    with open(summary_path) as f:
        summary = json.load(f)

    methods = [r["method"] for r in summary]
    metrics = ["rouge_l", "bertscore_f1", "cosine_similarity"]
    metric_labels = ["ROUGE-L", "BERTScore F1", "Cosine Similarity"]

    x = np.arange(len(metrics))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, row in enumerate(summary):
        values = [row.get(m, 0.0) for m in metrics]
        ax.bar(x + i * width, values, width, label=row["method"])

    ax.set_xticks(x + width)
    ax.set_xticklabels(metric_labels)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.set_title("LLM Impersonation – Method Comparison")
    ax.legend()
    plt.tight_layout()
    plt.show()